# ENERGIZE NILM Structured Pruning, Fine-tuning & Evaluation

This notebook supports **CNN** and **TCN** models and any PLEGMA appliance.

Pipeline:
1. **Configure** choose model, appliance and pruning ratio in one cell
2. **Baseline** load the trained checkpoint, measure cost, evaluate on test set
3. **Prune** apply global structured channel pruning (50% by default)
4. **Evaluate pruned** test-set metrics immediately after pruning
5. **Fine-tune** x-epoch recovery training on the training set
6. **Evaluate fine-tuned** test-set metrics after fine-tuning
7. **Export** save all results (Params, MACs, MB + metrics) to Excel

> Pruning functions live in `src_pytorch/pruner.py` and are imported here.

---

## Google Colab Setup
1. Upload your `ENERGIZE` folder to Google Drive
2. Run the cell below first and edit `DRIVE_PROJECT_PATH`

In [1]:
# ============================================================================
# COLAB SETUP â run this cell first
# ============================================================================
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_pruning', 'openpyxl'])

    # =========================================================================
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/ENERGIZE'  # <-- EDIT THIS
    # =========================================================================

    import os
    from pathlib import Path
    project_root = Path(DRIVE_PROJECT_PATH)
    if not project_root.exists():
        raise FileNotFoundError(f"Project folder not found: {project_root}")
    os.chdir(project_root)
    sys.path.insert(0, str(project_root))
    print(f"Project root: {project_root}")
else:
    import os
    from pathlib import Path
    project_root = Path(os.getcwd()).parent
    sys.path.insert(0, str(project_root))
    print(f"Running locally. Project root: {project_root}")

Running locally. Project root: c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch_pruning as tp
from tqdm import tqdm
from types import SimpleNamespace

# NILM package — models, data, evaluation, pipeline helpers
from src_pytorch import (
    CNN_NILM, TCN_NILM, GRU_NILM,
    SimpleNILMDataLoader,
    set_seeds, get_device, count_parameters,
    compute_status,
    compute_metrics,
    evaluate_model,
    save_pipeline_results,
    get_appliance_params,
    get_model_config,
)
from src_pytorch.pruner import (
    count_parameters_per_layer,
    get_model_stats,
    param_ratio_to_channel_ratio,
    apply_torch_pruning,
)

set_seeds(42)
device = get_device()

print(f"PyTorch version : {torch.__version__}")
print(f"torch_pruning   : {tp.__version__}")
print(f"Device          : {device}")

Seeds set to 42
CUDA not available, using CPU
PyTorch version : 2.10.0+cpu
torch_pruning   : 1.6.0
Device          : cpu


In [4]:
import src_pytorch
print(dir(src_pytorch))

['CALLBACKS', 'CNN_NILM', 'DATASET_CONFIGS', 'DATASET_SPLITS', 'DataLoaderNILM', 'EarlyStopping', 'GRU_NILM', 'MODEL_CONFIGS', 'ModelCheckpoint', 'NILMDataset', 'PLEGMA_PARAMS', 'REFIT_PARAMS', 'SimpleNILMDataLoader', 'SimpleTester', 'TCN_NILM', 'TRAINING', 'Tester', 'Trainer', 'TrainingHistory', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'apply_torch_pruning', 'compute_metrics', 'config', 'count_ops_and_params', 'count_parameters', 'count_parameters_per_layer', 'create_experiment_directories', 'data_loader', 'evaluate_model', 'get_appliance_params', 'get_dataset_config', 'get_dataset_split', 'get_device', 'get_model', 'get_model_config', 'get_model_stats', 'load_checkpoint', 'load_model', 'models', 'param_ratio_to_channel_ratio', 'print_model_summary', 'pruner', 'run_predictions', 'save_checkpoint', 'set_seeds', 'tester', 'train_model', 'trainer', 'utils']


## 2. Configuration

**Edit only this cell.** Everything else adapts automatically.

In [3]:
# ============================================================================
# USER CONFIGURATION
# ============================================================================
MODEL_NAME     = 'cnn'      # 'cnn'  |  'tcn'  |  'gru'
APPLIANCE_NAME = 'boiler'   # 'boiler'  |  'ac_1'  |  'washing_machine'
PRUNING_RATIO  = 0.75        # target *parameter* reduction  (0.0 – 1.0)
FINETUNE_LR    = 1e-4       # Adam learning rate for 1-epoch fine-tuning
DATASET_NAME   = 'plegma'

# ============================================================================
# AUTO-DERIVED — do not edit below this line
# ============================================================================
# Model config from src_pytorch (single source of truth)
_mcfg = get_model_config(MODEL_NAME)
cfg = {
    'window'          : _mcfg['input_window_length'],
    'batch_size'      : _mcfg['batch_size'],
    'depth'           : _mcfg.get('depth', 9),
    'filters'         : _mcfg.get('nb_filters'),
    'dropout'         : _mcfg.get('dropout', 0.2),
    'stacks'          : _mcfg.get('stacks', 1),
    'args_window_size': 1 if MODEL_NAME in ('cnn', 'gru') else _mcfg['input_window_length'],
}

# Appliance config from src_pytorch (single source of truth — synced with config.py)
app_cfg = get_appliance_params(DATASET_NAME, APPLIANCE_NAME)

WINDOW     = cfg['window']
BATCH_SIZE = cfg['batch_size']
THRESHOLD  = app_cfg['threshold']
CUTOFF     = app_cfg['cutoff']
MIN_ON     = app_cfg.get('min_on')
MIN_OFF    = app_cfg.get('min_off')
MAX_LENGTH = app_cfg.get('max_length')
pct        = int(PRUNING_RATIO * 100)

# Convert user-facing parameter ratio → internal channel ratio for MetaPruner.
_channel_ratio = param_ratio_to_channel_ratio(PRUNING_RATIO)

# ── Output directories ───────────────────────────────────────────────────────
OUT_DIR     = project_root / 'outputs' / f'{MODEL_NAME}_{APPLIANCE_NAME}'
MODELS_DIR  = OUT_DIR / 'models'       # model checkpoints (.pt files)
PREDS_DIR   = OUT_DIR / 'predictions'  # prediction vs ground-truth CSVs
METRICS_DIR = OUT_DIR / 'metrics'      # metrics CSVs
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH  = OUT_DIR / 'checkpoint' / 'model.pt'
DATA_DIR   = project_root / 'data' / 'processed' / DATASET_NAME / APPLIANCE_NAME

# Single comparative-results Excel for the whole pipeline
EXCEL_PATH = OUT_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_comparative_results.xlsx'

print(f"Model                : {MODEL_NAME.upper()}")
print(f"Appliance            : {APPLIANCE_NAME}")
print(f"Target param removal : {pct}%  →  channel ratio: {_channel_ratio:.4f}")
print(f"Fine-tune LR         : {FINETUNE_LR}")
print(f"Window length        : {WINDOW}")
print(f"Threshold            : {THRESHOLD} W  |  Cutoff: {CUTOFF} W")
print(f"Min ON               : {MIN_ON} samples  |  Min OFF: {MIN_OFF} samples")
if MAX_LENGTH:
    print(f"Max length           : {MAX_LENGTH} samples")
print(f"Checkpoint           : {CKPT_PATH}  ({'found' if CKPT_PATH.exists() else 'NOT FOUND'})")
print(f"Models dir           : {MODELS_DIR}")
print(f"Predictions dir      : {PREDS_DIR}")
print(f"Metrics dir          : {METRICS_DIR}")
print(f"Excel                : {EXCEL_PATH}")


Model                : CNN
Appliance            : boiler
Target param removal : 75%  →  channel ratio: 0.5000
Fine-tune LR         : 0.0001
Window length        : 299
Threshold            : 500 W  |  Cutoff: 4000 W
Min ON               : 30 samples  |  Min OFF: 6 samples
Checkpoint           : c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\cnn_boiler\checkpoint\model.pt  (NOT FOUND)
Models dir           : c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\cnn_boiler\models
Predictions dir      : c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\cnn_boiler\predictions
Metrics dir          : c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\cnn_boiler\metrics
Excel                : c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\cnn_

In [4]:
data_loader = SimpleNILMDataLoader(
    data_dir=str(DATA_DIR),
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    input_window_length=WINDOW,
    train=True,
    num_workers=0
)

print(f"Train batches : {len(data_loader.train)}")
print(f"Val   batches : {len(data_loader.val)}")
print(f"Test  batches : {len(data_loader.test)}")

Train batches : 23167
Val   batches : 1255
Test  batches : 1589


## 4. Helper Model Factory

A single function that builds and loads the correct architecture from a checkpoint.

In [5]:
def build_model(model_name: str, cfg: dict, ckpt_path, device) -> nn.Module:
    """Instantiate and load a NILM model from a checkpoint."""
    if model_name == 'cnn':
        model = CNN_NILM(input_window_length=cfg['window'])
    elif model_name == 'tcn':
        model = TCN_NILM(
            input_window_length=cfg['window'],
            depth=cfg['depth'],
            nb_filters=cfg['filters'],
            dropout=cfg['dropout'],
            stacks=cfg['stacks'],
        )
    elif model_name == 'gru':
        model = GRU_NILM(input_window_length=cfg['window'])
    else:
        raise ValueError(f"Unknown model: {model_name}. Choose 'cnn', 'gru', or 'tcn'.")
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return model.to(device).eval()


def evaluate_and_save(model, label, data_loader, model_name, cutoff, threshold,
                      device, input_window_length, preds_dir, prefix,
                      min_on=None, min_off=None, max_length=None):
    """
    Run test-set inference via evaluate_model, save predictions CSV, return metrics.

    Saved file: <preds_dir>/<prefix>_<label>_preds.csv
    Columns   : ground_truth_W, prediction_W[, ground_truth_status, predicted_status]
    """
    metrics, gt, pred, gt_status, pred_status = evaluate_model(
        model=model,
        data_loader=data_loader,
        model_name=model_name,
        cutoff=cutoff,
        threshold=threshold,
        device=device,
        input_window_length=input_window_length,
        min_on=min_on,
        min_off=min_off,
        max_length=max_length,
    )

    pred_data = {'ground_truth_W': gt, 'prediction_W': pred}
    if gt_status is not None:
        pred_data['ground_truth_status'] = gt_status.astype(int)
        pred_data['predicted_status']    = pred_status.astype(int)

    csv_path = preds_dir / f'{prefix}_{label}_preds.csv'
    pd.DataFrame(pred_data).to_csv(csv_path, index=False)
    print(f"  Predictions saved : {csv_path.name}  (columns: {list(pred_data.keys())})")
    return metrics


print("build_model() and evaluate_and_save() defined.")


build_model() and evaluate_and_save() defined.


## 5. Baseline Load & Evaluate

In [7]:
# Load baseline model
baseline_model = build_model(MODEL_NAME, cfg, CKPT_PATH, device)
dummy_input    = torch.randn(1, WINDOW).to(device)

baseline_params, baseline_macs, baseline_mb = get_model_stats(baseline_model, dummy_input)

print(f"Model      : {MODEL_NAME.upper()}")
print(f"Parameters : {baseline_params:,}")
print(f"MACs       : {baseline_macs:,}")
print(f"Size       : {baseline_mb:.3f} MB")

Model      : CNN
Parameters : 13,863,449
MACs       : 24,179,293.0
Size       : 52.885 MB


In [14]:
_prefix = f'{MODEL_NAME}_{APPLIANCE_NAME}'

print(f"Evaluating {MODEL_NAME.upper()} baseline on test set...")
baseline_metrics = evaluate_and_save(
    model=baseline_model,
    label='baseline',
    data_loader=data_loader,
    model_name=MODEL_NAME,
    cutoff=CUTOFF,
    threshold=THRESHOLD,
    device=device,
    input_window_length=WINDOW,
    preds_dir=PREDS_DIR,
    prefix=_prefix,
    min_on=MIN_ON,
    min_off=MIN_OFF,
)

print(f"\nBaseline Results:")
for k, v in baseline_metrics.items():
    print(f"  {k:<25}: {v:.4f}")

Evaluating CNN baseline on test set...


Inference: 100%|██████████| 1589/1589 [07:05<00:00,  3.73it/s]


  Predictions saved : cnn_boiler_baseline_preds.csv  (columns: ['ground_truth_W', 'prediction_W', 'ground_truth_status', 'predicted_status'])

Baseline Results:
  mae                      : 8.0491
  f1                       : 0.9000
  accuracy                 : 0.9962
  precision                : 0.8276
  recall                   : 0.9863
  tp                       : 27583.0000
  tn                       : 1593127.0000
  fp                       : 5746.0000
  fn                       : 382.0000
  total_gt_energy_wh       : 22142.4775
  total_pred_energy_wh     : 22805.2051
  energy_error_percent     : 2.9930
  f1_complex               : 0.9177


## 6. Prune & Evaluate

In [15]:
# Reload a fresh instance â pruning is irreversible
pruned_model = build_model(MODEL_NAME, cfg, CKPT_PATH, device)
pruning_args = SimpleNamespace(window_size=cfg['args_window_size'])

pruned_model, _ = apply_torch_pruning(
    model=pruned_model,
    args=pruning_args,
    inputs=dummy_input,
    pruning_ratio=PRUNING_RATIO,
)

Baseline model  — MACs: 24,179,293.0  |  Params: 13,863,449
Pruned model    — MACs: 6,009,343.0  |  Params: 3,466,176
MACs reduction  : 75.1%
Param reduction : 75.0%
Output shape    : torch.Size([1, 1])


In [16]:
# Model cost after pruning
pruned_params, pruned_macs, pruned_mb = get_model_stats(pruned_model, dummy_input)

print(f"Parameters : {pruned_params:,}  ({(1 - pruned_params/baseline_params)*100:.1f}% reduction)")
print(f"MACs       : {pruned_macs:,}  ({(1 - pruned_macs/baseline_macs)*100:.1f}% reduction)")
print(f"Size       : {pruned_mb:.3f} MB  ({(1 - pruned_mb/baseline_mb)*100:.1f}% reduction)")

print("\nPer-layer parameter counts (after pruning):")
for name, cnt in count_parameters_per_layer(pruned_model).items():
    print(f"  {name:<60} {cnt:>10,}")

Parameters : 3,466,176  (75.0% reduction)
MACs       : 6,009,343.0  (75.1% reduction)
Size       : 13.222 MB  (75.0% reduction)

Per-layer parameter counts (after pruning):
  network.0                                                           198
  network.2                                                         2,030
  network.4                                                         1,700
  network.6                                                         2,323
  network.9                                                         2,900
  network.13                                                    3,456,512
  network.16                                                          513


In [17]:
print(f"Evaluating pruned {MODEL_NAME.upper()} on test set...")
pruned_metrics = evaluate_and_save(
    model=pruned_model,
    label=f'pruned_{pct}pct',
    data_loader=data_loader,
    model_name=MODEL_NAME,
    cutoff=CUTOFF,
    threshold=THRESHOLD,
    device=device,
    input_window_length=WINDOW,
    preds_dir=PREDS_DIR,
    prefix=_prefix,
    min_on=MIN_ON,
    min_off=MIN_OFF,
)

print(f"\nPruned Results:")
for k, v in pruned_metrics.items():
    print(f"  {k:<25}: {v:.4f}")

Evaluating pruned CNN on test set...


Inference: 100%|██████████| 1589/1589 [02:46<00:00,  9.55it/s]


  Predictions saved : cnn_boiler_pruned_75pct_preds.csv  (columns: ['ground_truth_W', 'prediction_W', 'ground_truth_status', 'predicted_status'])

Pruned Results:
  mae                      : 48.9987
  f1                       : 0.0000
  accuracy                 : 0.9828
  precision                : 0.0000
  recall                   : 0.0000
  tp                       : 0.0000
  tn                       : 1598873.0000
  fp                       : 0.0000
  fn                       : 27965.0000
  total_gt_energy_wh       : 22142.4775
  total_pred_energy_wh     : 0.0000
  energy_error_percent     : 100.0000
  f1_complex               : 0.0000


In [19]:
pruned_ckpt = MODELS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct.pt'
torch.save(pruned_model.state_dict(), pruned_ckpt)
print(f"Pruned checkpoint saved: {pruned_ckpt}")

Pruned checkpoint saved: c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\cnn_boiler\models\cnn_boiler_pruned_75pct.pt


## 7. Fine-tune

`FINETUNE_EPOCHS` epochs of MSE training on the training set with Adam at `FINETUNE_LR`.
The pruned model structure is preserved — no re-growing of pruned channels.

In [20]:
FINETUNE_EPOCHS =1

In [ ]:
optimizer = torch.optim.Adam(pruned_model.parameters(), lr=FINETUNE_LR)
loss_fn   = nn.MSELoss()

for epoch in range(1, FINETUNE_EPOCHS + 1):
    pruned_model.train()
    total_loss = 0.0
    total_mae  = 0.0
    n_batches  = 0

    for batch_x, batch_y in tqdm(data_loader.train, desc=f"Fine-tuning epoch {epoch}/{FINETUNE_EPOCHS}"):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = pruned_model(batch_x)

        # Align target shape to model output
        # CNN/GRU : output (B, 1),          target (B,)         → unsqueeze to (B, 1)
        # TCN     : output (B, seq_len, 1), target (B, seq_len) → unsqueeze to (B, seq_len, 1)
        if MODEL_NAME in ('cnn', 'gru') and batch_y.dim() == 1:
            batch_y = batch_y.unsqueeze(1)
        elif MODEL_NAME == 'tcn' and batch_y.dim() == 2:
            batch_y = batch_y.unsqueeze(-1)

        loss = loss_fn(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_mae  += torch.mean(torch.abs(outputs.detach() - batch_y)).item()
        n_batches  += 1

    print(f"Epoch {epoch}/{FINETUNE_EPOCHS}  avg MSE loss: {total_loss/n_batches:.6f}  "
          f"avg MAE: {total_mae/n_batches:.6f}")

pruned_model.eval()
print("\nFine-tuning complete.")

In [22]:
finetuned_ckpt = MODELS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_finetuned.pt'
torch.save(pruned_model.state_dict(), finetuned_ckpt)
print(f"Fine-tuned checkpoint saved: {finetuned_ckpt}")

Fine-tuned checkpoint saved: c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\tcn_boiler\models\tcn_boiler_pruned_75pct_finetuned.pt


## 8. Evaluate Fine-tuned Model

Architecture (and therefore Params / MACs / MB) is unchanged by fine-tuning.

In [ ]:
finetuned_params, finetuned_macs, finetuned_mb = get_model_stats(pruned_model, dummy_input)

print(f"Evaluating fine-tuned {MODEL_NAME.upper()} on test set...")
finetuned_metrics = evaluate_and_save(
    model=pruned_model,
    label=f'pruned_{pct}pct_finetuned',
    data_loader=data_loader,
    model_name=MODEL_NAME,
    cutoff=CUTOFF,
    threshold=THRESHOLD,
    device=device,
    input_window_length=WINDOW,
    preds_dir=PREDS_DIR,
    prefix=_prefix,
    min_on=MIN_ON,
    min_off=MIN_OFF,
)

print(f"\nFine-tuned Results:")
for k, v in finetuned_metrics.items():
    print(f"  {k:<25}: {v:.4f}")

## 9. Export Results to Excel

In [ ]:
def make_row(label, pruning_pct, params, macs, mb, metrics):
    return {
        'Model'           : label,
        'Architecture'    : MODEL_NAME.upper(),
        'Appliance'       : APPLIANCE_NAME,
        'Pruning_Ratio_%' : pruning_pct,
        'Params'          : params,
        'MACs'            : macs,
        'MB'              : mb,
        'MAE'             : round(metrics['mae'],                  4),
        'F1'              : round(metrics['f1'],                   4),
        'F1_Complex'      : round(metrics['f1_complex'],           4) if 'f1_complex' in metrics else '',
        'Precision'       : round(metrics['precision'],            4),
        'Recall'          : round(metrics['recall'],               4),
        'Accuracy'        : round(metrics['accuracy'],             4),
        'Energy_Error_%'  : round(metrics['energy_error_percent'], 2),
    }


results_df = pd.DataFrame([
    make_row(f'{MODEL_NAME.upper()} Baseline',
             0, baseline_params, baseline_macs, baseline_mb, baseline_metrics),
    make_row(f'{MODEL_NAME.upper()} Pruned {pct}%',
             pct, pruned_params, pruned_macs, pruned_mb, pruned_metrics),
    make_row(f'{MODEL_NAME.upper()} Pruned {pct}% + Fine-tuned {FINETUNE_EPOCHS}ep',
             pct, finetuned_params, finetuned_macs, finetuned_mb, finetuned_metrics),
])

# Load existing Excel and upsert rows (so notebook 04 entries are preserved)
if EXCEL_PATH.exists():
    existing = pd.read_excel(EXCEL_PATH)
    known    = set(results_df['Model'])
    updated  = pd.concat(
        [existing[~existing['Model'].isin(known)], results_df],
        ignore_index=True,
    )
else:
    updated = results_df

updated.to_excel(EXCEL_PATH, index=False)
print(f"Results saved to: {EXCEL_PATH}")
updated

In [ ]:
# Save pipeline metrics CSV to metrics/ folder (mirrors main.py behaviour)
save_pipeline_results(
    rows=[
        {**baseline_metrics,  'label': f'{MODEL_NAME.upper()} Baseline'},
        {**pruned_metrics,    'label': f'{MODEL_NAME.upper()} Pruned {pct}%'},
        {**finetuned_metrics, 'label': f'{MODEL_NAME.upper()} Pruned {pct}% + Fine-tuned 1ep'},
    ],
    output_dir=METRICS_DIR,
    appliance=APPLIANCE_NAME,
    model_name=MODEL_NAME,
)


# Save pipeline metrics CSV to metrics/ folder (mirrors main.py behaviour)
save_pipeline_results(
    rows=[
        {**baseline_metrics,  'label': f'{MODEL_NAME.upper()} Baseline'},
        {**pruned_metrics,    'label': f'{MODEL_NAME.upper()} Pruned {pct}%'},
        {**finetuned_metrics, 'label': f'{MODEL_NAME.upper()} Pruned {pct}% + Fine-tuned {FINETUNE_EPOCHS}ep'},
    ],
    output_dir=METRICS_DIR,
    appliance=APPLIANCE_NAME,
    model_name=MODEL_NAME,
)

In [ ]:
C, W3 = 22, 16
SEP   = "=" * (C + W3 * 3 + 3)
sep   = "-" * (C + W3 * 3 + 3)
hfmt  = f"{{:<{C}}} {{:>{W3}}} {{:>{W3}}} {{:>{W3}}}"
rfmt  = f"{{:<{C}}} {{:>{W3}}} {{:>{W3}}} {{:>{W3}}}"

col_baseline  = f"{MODEL_NAME.upper()} Baseline"
col_pruned    = f"Pruned {pct}%"
col_finetuned = f"Pruned+FT {FINETUNE_EPOCHS}ep"

print(SEP)
print(f"PRUNING SUMMARY  |  {MODEL_NAME.upper()}  |  {APPLIANCE_NAME}  |  pruning={pct}%  |  LR={FINETUNE_LR}")
print(SEP)
print(hfmt.format('Metric', col_baseline, col_pruned, col_finetuned))
print(sep)

def rrow(label, b, p, f):
    print(rfmt.format(label, b, p, f))

rrow('Params',        f"{baseline_params:,}",   f"{pruned_params:,}",   f"{finetuned_params:,}")
rrow('MACs',          f"{baseline_macs:,}",     f"{pruned_macs:,}",     f"{finetuned_macs:,}")
rrow('MB',            f"{baseline_mb:.3f}",     f"{pruned_mb:.3f}",     f"{finetuned_mb:.3f}")
print(sep)
rrow('MAE (W)',       f"{baseline_metrics['mae']:.4f}",
                      f"{pruned_metrics['mae']:.4f}",
                      f"{finetuned_metrics['mae']:.4f}")
rrow('F1',            f"{baseline_metrics['f1']:.4f}",
                      f"{pruned_metrics['f1']:.4f}",
                      f"{finetuned_metrics['f1']:.4f}")
rrow('Precision',     f"{baseline_metrics['precision']:.4f}",
                      f"{pruned_metrics['precision']:.4f}",
                      f"{finetuned_metrics['precision']:.4f}")
rrow('Recall',        f"{baseline_metrics['recall']:.4f}",
                      f"{pruned_metrics['recall']:.4f}",
                      f"{finetuned_metrics['recall']:.4f}")
rrow('Accuracy',      f"{baseline_metrics['accuracy']:.4f}",
                      f"{pruned_metrics['accuracy']:.4f}",
                      f"{finetuned_metrics['accuracy']:.4f}")
rrow('Energy Err %',  f"{baseline_metrics['energy_error_percent']:.2f}",
                      f"{pruned_metrics['energy_error_percent']:.2f}",
                      f"{finetuned_metrics['energy_error_percent']:.2f}")
print(SEP)
print(f"Excel  : {EXCEL_PATH}")
print(f"Models : {MODELS_DIR}")
print(f"  {pruned_ckpt.name}")
print(f"  {finetuned_ckpt.name}")
print(f"Preds  : {PREDS_DIR}")
print(SEP)